In [8]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import matplotlib


# Set matplotlib to use high-quality settings
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']


# Function to parse mean ± std format
def parse_mean_std(value_str):
    parts = value_str.split('±')
    mean = float(parts[0].strip())
    std = float(parts[1].strip())
    return mean, std


# Read the CSV files
corpus_df = pd.read_csv('corpus_scores_summary_formatted.csv')
reg_df = pd.read_csv('reg_scores_summary_formatted.csv')


# Get models and metrics
desired_order = ['HistGen', 'UNI', 'Conch', 'UNI2', 'TITAN']
models = [m for m in desired_order if m in corpus_df['Model'].tolist()]
#metrics = ['BLEU-1', 'BLEU-2', 'BLEU-3', 'BLEU-4', 'METEOR', 'ROUGE-L', 'REG']
metrics = ['BLEU-4', 'METEOR', 'ROUGE-L', 'REG']

# Display names for axes (REG -> REGScore)
metrics_display = ['BLEU-4', 'METEOR', 'ROUGE-L', 'REGScore']


# Define color palette (EXACT SAME for both Plotly and Matplotlib)
model_colors = {
    'HistGen': '#7f8c8d',
    'UNI': '#3498db',
    'UNI2': '#8e44ad',
    'TITAN': '#e74c3c',
    'Conch': '#16b895'
}


# Extract data for each model
plot_data = {model: {'means': [], 'stds': []} for model in models}

for model in models:
    for metric in metrics[:-1]:
        value_str = corpus_df.loc[corpus_df['Model'] == model, metric].values[0]
        mean, std = parse_mean_std(value_str)
        plot_data[model]['means'].append(mean)
        plot_data[model]['stds'].append(std)

    reg_value_str = reg_df.loc[reg_df['Model'] == model, 'REG'].values[0]
    mean, std = parse_mean_std(reg_value_str)
    plot_data[model]['means'].append(mean)
    plot_data[model]['stds'].append(std)


print("Data loaded successfully!")
print(f"Models: {models}")
print(f"Metrics: {metrics}\n")


# Store all Plotly figures for display
all_plotly_figures = []


# ============================================================
# PART 1: Individual plots for each model
# Dot plot + error bars
# ============================================================

for model in models:
    means = np.array(plot_data[model]['means'])
    stds = np.array(plot_data[model]['stds'])
    color = model_colors[model]
    x_pos = np.arange(len(metrics))

    # --- PLOTLY VERSION (Interactive) ---
    fig_plotly = go.Figure()

    fig_plotly.add_trace(go.Scatter(
        x=metrics_display,
        y=means,
        mode='markers',
        name=model,
        marker=dict(
            size=12,
            color=color,
            line=dict(color='white', width=2),
            symbol='circle'
        ),
        error_y=dict(
            type='data',
            array=stds,
            visible=True,
            thickness=1.6,
            width=4
        ),
        hovertemplate='<b>%{x}</b><br>Score: %{y:.4f}<br>±%{customdata:.4f}<extra></extra>',
        customdata=stds
    ))

    fig_plotly.update_layout(
        title=dict(
            text=f'<b>{model} Performance Across Metrics (5 Seeds)</b>',
            font=dict(size=16),
            x=0.5,
            xanchor='center'
        ),
        xaxis=dict(
            title=dict(text='<b>Metrics</b>', font=dict(size=14)),
            tickfont=dict(size=12),
            type='category',
            showgrid=True,
            gridwidth=1,
            gridcolor='rgba(128,128,128,0.2)'
        ),
        yaxis=dict(
            title=dict(text='<b>Score</b>', font=dict(size=14)),
            tickfont=dict(size=12),
            range=[0.4, 0.8],
            showgrid=True,
            gridwidth=1,
            gridcolor='rgba(128,128,128,0.2)'
        ),
        width=1200,
        height=600,
        margin=dict(l=80, r=40, t=80, b=80, pad=0),
        hovermode='x',
        plot_bgcolor='white',
        paper_bgcolor='white',
        legend=dict(
            x=0.02,
            y=0.02,
            bgcolor='rgba(255,255,255,0.8)',
            bordercolor='black',
            borderwidth=1,
            font=dict(size=11)
        )
    )

    fig_plotly.write_html(f'model_{model.lower()}_individual_dot.html')
    all_plotly_figures.append((model, fig_plotly))

    # --- MATPLOTLIB VERSION (High-quality PDF/PNG) ---
    fig_mpl, ax = plt.subplots(figsize=(12, 6))

    ax.errorbar(
        x_pos,
        means,
        yerr=stds,
        fmt='o',
        color=color,
        ecolor='black',
        elinewidth=1.5,
        capsize=5,
        capthick=1.5,
        markersize=10,
        markerfacecolor=color,
        markeredgecolor='white',
        markeredgewidth=2,
        linestyle='none',
        label=model
    )

    # Styling
    ax.set_xlabel('Metrics', fontsize=14, fontweight='bold')
    ax.set_ylabel('Score', fontsize=14, fontweight='bold')
    ax.set_title(f'{model} Performance Across Metrics',
                 fontsize=16, fontweight='bold', pad=20)

    ax.set_xticks(x_pos)
    ax.set_xticklabels(metrics_display, fontsize=12)
    ax.set_ylim(0.4, 0.8)
    ax.set_yticks(np.arange(0.4, 0.85, 0.05))
    ax.tick_params(axis='y', labelsize=11)

    ax.grid(True, axis='y', alpha=0.3, linestyle='--', linewidth=0.5)
    ax.set_axisbelow(True)
    ax.set_facecolor('white')

    ax.legend(loc='lower left', fontsize=11, framealpha=0.95,
              edgecolor='black', fancybox=False)

    plt.tight_layout()

    # Save high-quality PDF and PNG
    plt.savefig(f'model_{model.lower()}_individual_dot.pdf', dpi=300, bbox_inches='tight')
    plt.savefig(f'model_{model.lower()}_individual_dot.png', dpi=300, bbox_inches='tight')
    plt.close()

    print(f"✓ Created dot plots for {model}")


# ============================================================
# PART 2: Combined overview plot (all models)
# Grouped dot plot + error bars
# ============================================================

# --- PLOTLY VERSION (Interactive) ---
fig_overview_plotly = go.Figure()

n_models = len(models)
offsets = np.linspace(-0.24, 0.24, n_models)

for i, model in enumerate(models):
    means = np.array(plot_data[model]['means'])
    stds = np.array(plot_data[model]['stds'])
    color = model_colors[model]

    fig_overview_plotly.add_trace(go.Scatter(
        x=metrics_display,
        y=means,
        mode='markers',
        name=model,
        marker=dict(
            size=11,
            color=color,
            line=dict(color='white', width=1.8),
            symbol='circle'
        ),
        error_y=dict(
            type='data',
            array=stds,
            visible=True,
            thickness=1.4,
            width=4
        ),
        hovertemplate='<b>%{fullData.name}</b><br>Metric: %{x}<br>Score: %{y:.4f}<br>±%{customdata:.4f}<extra></extra>',
        customdata=stds,
        x0=0
    ))

fig_overview_plotly.update_layout(
    title=dict(
        text='<b>Multi-Metric Performance Comparison Across All Models (5 Seeds)</b>',
        font=dict(size=16),
        x=0.5,
        xanchor='center'
    ),
    xaxis=dict(
        title=dict(text='<b>Metrics</b>', font=dict(size=14)),
        tickfont=dict(size=12),
        type='category',
        showgrid=True,
        gridwidth=1,
        gridcolor='rgba(128,128,128,0.2)'
    ),
    yaxis=dict(
        title=dict(text='<b>Score</b>', font=dict(size=14)),
        tickfont=dict(size=12),
        range=[0.4, 0.8],
        showgrid=True,
        gridwidth=1,
        gridcolor='rgba(128,128,128,0.2)'
    ),
    width=1300,
    height=650,
    margin=dict(l=80, r=40, t=80, b=80, pad=0),
    hovermode='closest',
    plot_bgcolor='white',
    paper_bgcolor='white',
    legend=dict(
        x=0.02,
        y=0.02,
        bgcolor='rgba(255,255,255,0.8)',
        bordercolor='black',
        borderwidth=1,
        font=dict(size=11)
    )
)

fig_overview_plotly.write_html('all_models_overview_grouped_dot.html')
all_plotly_figures.append(('Overview', fig_overview_plotly))


# --- MATPLOTLIB VERSION (High-quality PDF/PNG) ---
fig_mpl, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(metrics))
n_models = len(models)
offsets = np.linspace(-0.24, 0.24, n_models)

for i, model in enumerate(models):
    means = np.array(plot_data[model]['means'])
    stds = np.array(plot_data[model]['stds'])
    color = model_colors[model]

    ax.errorbar(
        x + offsets[i],
        means,
        yerr=stds,
        fmt='o',
        color=color,
        ecolor='black',
        elinewidth=1.2,
        capsize=4,
        capthick=1.2,
        markersize=9,
        markerfacecolor=color,
        markeredgecolor='white',
        markeredgewidth=1.8,
        linestyle='none',
        label=model
    )

# Styling
ax.set_xlabel('Metrics', fontsize=14, fontweight='bold')
ax.set_ylabel('Score', fontsize=14, fontweight='bold')
#ax.set_title('Multi-Metric Performance Comparison Across All Models', fontsize=16, fontweight='bold', pad=20)

ax.set_xticks(x)
ax.set_xticklabels(metrics_display, fontsize=12)
for lbl in ax.get_xticklabels():
    lbl.set_fontweight("bold")
ax.set_ylim(0.4, 0.8)
ax.set_yticks(np.arange(0.4, 0.85, 0.05))
ax.tick_params(axis='y', labelsize=12)
for lbl in ax.get_yticklabels():
    lbl.set_fontweight("bold")

ax.grid(True, axis='y', alpha=0.3, linestyle='--', linewidth=0.5)
ax.set_axisbelow(True)
ax.set_facecolor('white')

ax.legend(loc='lower left', fontsize=12, framealpha=0.95,
          edgecolor='black', fancybox=False, prop={"weight": "bold"})

plt.tight_layout()

plt.savefig('all_models_overview_grouped_dot.pdf', dpi=300, bbox_inches='tight')
plt.savefig('all_models_overview_grouped_dot.png', dpi=300, bbox_inches='tight')
plt.close()

print("\n✓ Created combined grouped dot plot")


# ============================================================
# PART 3: Display all Plotly figures in Jupyter notebook
# ============================================================

print("\n" + "="*70)
print("DISPLAYING ALL INTERACTIVE FIGURES IN JUPYTER")
print("="*70)

for model_name, figure in all_plotly_figures:
    print(f"\nShowing: {model_name}")
    figure.show()

print("\n" + "="*70)
print("SUMMARY OF GENERATED FILES:")
print("="*70)
print("\nIndividual Model Plots:")
for model in models:
    print(f"  {model}:")
    print(f"    - model_{model.lower()}_individual_dot.html (Plotly - interactive)")
    print(f"    - model_{model.lower()}_individual_dot.pdf (Matplotlib - THESIS)")
    print(f"    - model_{model.lower()}_individual_dot.png (Matplotlib - backup)")

print("\nCombined Overview Plot:")
print("    - all_models_overview_grouped_dot.html (Plotly - interactive)")
print("    - all_models_overview_grouped_dot.pdf (Matplotlib - THESIS)")
print("    - all_models_overview_grouped_dot.png (Matplotlib - backup)")
print("="*70)
print("\n✓ Plotly: Interactive HTML with hover tooltips")
print("✓ Matplotlib: Vector PDF (300 DPI) - paper-ready")
print("✓ Grouped dot plot: better for discrete metrics")
print("✓ Error bars preserve std across 5 seeds")
print("="*70)

Data loaded successfully!
Models: ['HistGen', 'UNI', 'Conch', 'UNI2', 'TITAN']
Metrics: ['BLEU-4', 'METEOR', 'ROUGE-L', 'REG']

✓ Created dot plots for HistGen
✓ Created dot plots for UNI
✓ Created dot plots for Conch
✓ Created dot plots for UNI2
✓ Created dot plots for TITAN

✓ Created combined grouped dot plot

DISPLAYING ALL INTERACTIVE FIGURES IN JUPYTER

Showing: HistGen



Showing: UNI



Showing: Conch



Showing: UNI2



Showing: TITAN



Showing: Overview



SUMMARY OF GENERATED FILES:

Individual Model Plots:
  HistGen:
    - model_histgen_individual_dot.html (Plotly - interactive)
    - model_histgen_individual_dot.pdf (Matplotlib - THESIS)
    - model_histgen_individual_dot.png (Matplotlib - backup)
  UNI:
    - model_uni_individual_dot.html (Plotly - interactive)
    - model_uni_individual_dot.pdf (Matplotlib - THESIS)
    - model_uni_individual_dot.png (Matplotlib - backup)
  Conch:
    - model_conch_individual_dot.html (Plotly - interactive)
    - model_conch_individual_dot.pdf (Matplotlib - THESIS)
    - model_conch_individual_dot.png (Matplotlib - backup)
  UNI2:
    - model_uni2_individual_dot.html (Plotly - interactive)
    - model_uni2_individual_dot.pdf (Matplotlib - THESIS)
    - model_uni2_individual_dot.png (Matplotlib - backup)
  TITAN:
    - model_titan_individual_dot.html (Plotly - interactive)
    - model_titan_individual_dot.pdf (Matplotlib - THESIS)
    - model_titan_individual_dot.png (Matplotlib - backup)

Combined 